# MVP 11 Sources Test (Production Pipeline)

This notebook validates the **11-source MVP** using real extractors.

It will:
1. Install dependencies
2. Authenticate with Google Earth Engine (GEE)
3. Run extraction for all sources
4. Visualize sample outputs
5. Save outputs to `/content/`
6. Show data quality metrics


In [ ]:
# 1) Install dependencies
!pip -q install earthengine-api geemap geopandas shapely pandas pyarrow requests lxml beautifulsoup4 matplotlib

In [ ]:
# 2) Mount code path (if needed)
import os, sys

# If running in Colab with repo copied to /content/production_pipeline
repo_root = '/content'
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

print('Python path ready')

In [ ]:
# 3) GEE setup (RECOMMENDED)
from production_pipeline.setup_colab import setup_gee_for_colab

# Set your GEE cloud project here
setup_gee_for_colab('genai-bangladesh-drought-demo')

print('✅ Ready to extract!')


In [ ]:
# 4) Imports
from datetime import datetime, timedelta
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
from shapely.geometry import box

from production_pipeline.extractors.gee_extractor import (
    extract_chirps_daily,
    extract_smap_soil_moisture,
    extract_era5_temperature,
    extract_modis_ndvi,
)
from production_pipeline.extractors.climate_drivers import scrape_enso_indices
from production_pipeline.extractors.bmd_extractor import scrape_spi_tables
from production_pipeline.extractors.bamis_extractor import scrape_crop_calendars
from production_pipeline.extractors.static_extractor import (
    load_hdx_boundaries,
    load_mapspam_crops,
    load_soilgrids_properties,
)
from production_pipeline.processors.spi_spei_calculator import calculate_spi, calculate_spei

In [ ]:
# 5) Build/load district geometries

def build_fallback_districts_gdf():
    print('⚠️  WARNING: Using SYNTHETIC fallback districts (3 only)!')
    print('⚠️  This is OK for quick API testing but NOT for historical downloads.')
    print('⚠️  For real analysis, ensure hdx_boundaries.csv loads 64 districts.')
    rows = [
        {'district_id': 'BD-DHK', 'district_name': 'Dhaka', 'geometry': box(90.25, 23.60, 90.55, 23.95)},
        {'district_id': 'BD-RJS', 'district_name': 'Rajshahi', 'geometry': box(88.45, 24.20, 88.85, 24.55)},
        {'district_id': 'BD-KHL', 'district_name': 'Khulna', 'geometry': box(89.30, 22.70, 89.75, 23.10)},
    ]
    return gpd.GeoDataFrame(rows, geometry='geometry', crs='EPSG:4326')

try:
    hdx_df = load_hdx_boundaries()
    if hasattr(hdx_df, 'empty') and (not hdx_df.empty) and ('geometry' in hdx_df.columns):
        districts = hdx_df
    else:
        districts = build_fallback_districts_gdf()
except Exception:
    districts = build_fallback_districts_gdf()

districts.head()

In [ ]:
# 6) Run all extractors (1-week window)
end_date = datetime.now()
start_date = end_date - timedelta(days=7)

start_str = start_date.strftime('%Y-%m-%d')
end_str = end_date.strftime('%Y-%m-%d')

out_dir = Path('/content')
out_dir.mkdir(parents=True, exist_ok=True)

results = {}

results['chirps'] = extract_chirps_daily(start_str, end_str, districts, str(out_dir / 'chirps_daily.parquet'))
results['smap'] = extract_smap_soil_moisture(start_str, end_str, districts, str(out_dir / 'smap_soil_moisture.parquet'))
results['era5'] = extract_era5_temperature(start_str, end_str, districts, str(out_dir / 'era5_temperature.parquet'))
results['modis'] = extract_modis_ndvi(start_str, end_str, districts, str(out_dir / 'modis_ndvi.parquet'))
results['enso'] = scrape_enso_indices()
results['bmd_spi'] = scrape_spi_tables()
results['bamis_crops'] = scrape_crop_calendars()
results['hdx'] = load_hdx_boundaries()
results['mapspam'] = load_mapspam_crops()
results['soilgrids'] = load_soilgrids_properties()

# computed SPI/SPEI (11-source MVP includes computed indicators)
spi_input = results['chirps'][['district_id', 'date', 'rainfall_mm']].rename(columns={'date': 'record_date'})
spi_input['pet_mm'] = (spi_input['rainfall_mm'].fillna(0) * 0.6).clip(lower=0)
results['spi'] = calculate_spi(spi_input, value_col='rainfall_mm', group_cols=['district_id'], window=3)
results['spei'] = calculate_spei(spi_input, precip_col='rainfall_mm', pet_col='pet_mm', group_cols=['district_id'], window=3)

for k, df in results.items():
    print(f"✅ {k}: {len(df)} rows")

In [ ]:
# 7) Save CSV copies to /content/
for name, df in results.items():
    csv_path = out_dir / f'{name}.csv'
    df.to_csv(csv_path, index=False)
    print('Saved:', csv_path)

In [ ]:
# 8) Data quality metrics summary
quality_rows = []
for name, df in results.items():
    quality_col = None
    for c in ['data_quality', 'quality_flag']:
        if c in df.columns:
            quality_col = c
            break

    quality_rows.append({
        'dataset': name,
        'rows': len(df),
        'columns': len(df.columns),
        'null_pct_total': round(float(df.isna().sum().sum()) / max(1, df.size) * 100, 2),
        'quality_col': quality_col,
        'quality_counts': dict(df[quality_col].value_counts(dropna=False)) if quality_col else {},
    })

quality_df = pd.DataFrame(quality_rows)
quality_df

In [ ]:
# 9) Quick visualization examples
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Rainfall trend
plot_df = results['chirps'].copy()
plot_df['date'] = pd.to_datetime(plot_df['date'])
plot_df.groupby('date')['rainfall_mm'].mean().plot(ax=axes[0], marker='o', title='Mean District Rainfall (CHIRPS)')
axes[0].set_ylabel('mm/day')

# ENSO trend
enso_df = results['enso'].copy()
enso_df['date'] = pd.to_datetime(enso_df['date'])
enso_df.set_index('date')[['oni_value', 'mei_value', 'soi_value']].plot(ax=axes[1], title='ENSO Indices (Last 12 Months)')
axes[1].set_ylabel('Index value')

plt.tight_layout()
plt.show()

## Done

All extraction outputs are saved under `/content/`.

For production deployment, run the same extractors inside your scheduled pipeline job and load results into BigQuery/ArangoDB.
